# Tree accuracy

Run by the `tree_accuracy` rule after the tree and the taxonomy are built (one notebook per
`{derep}` x `{copies}` tree; `tree_comparison` compares them). Parameters come from the JSON file
named by `NB_PARAMS`. All monophyly numbers are **rooting-independent** (see
`scripts/tree_accuracy_lib.py`); support values are FastTree's SH-like values.

In [ ]:
import json, os, sys
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
%matplotlib inline
from Bio import Phylo

P = json.load(open(os.environ["NB_PARAMS"]))
sys.path.insert(0, P["scripts_dir"])
import tree_accuracy_lib as lib
OUT = Path(P["out_dir"]); OUT.mkdir(parents=True, exist_ok=True)
print("tree:", P["label"], "|", P["tree"])
tree = Phylo.read(P["tree"], "newick")
taxonomy = lib.read_taxonomy(P["taxonomy"])
n_tips = tree.count_terminals()
labelled = sum(1 for t in tree.get_terminals() if t.name in taxonomy)
print(f"{n_tips} tips, {labelled} with a database taxonomy")

## 1. Clade fragmentation (unrooted)

A name with more than one *piece* is not monophyletic on this tree.

In [ ]:
pieces = {rank: lib.pieces_unrooted(tree, taxonomy, rank) for rank in lib.RANKS}
summary, names = lib.fragmentation(tree, taxonomy, pieces_by_rank=pieces)
print(f"{'rank':8}{'names':>7}{'fragmented':>12}{'extra pieces':>14}{'tips in frag.':>15}")
for r in summary:
    print(f"{r['rank']:8}{r['names']:>7}{r['fragmented_names']:>12}{r['extra_pieces']:>14}{r['tips_in_fragmented']:>15}")
with open(OUT / "fragmentation.tsv", "w") as f:
    f.write("rank\tname\ttips\tpieces\textra_piece_sizes\n")
    for n in names:
        f.write(f"{n['rank']}\t{n['name']}\t{n['tips']}\t{n['pieces']}\t{','.join(map(str, n['extra_sizes']))}\n")
for rank in ("phylum", "class", "order"):
    worst = sorted([n for n in names if n["rank"] == rank and n["pieces"] > 1], key=lambda n: -n["pieces"])[:8]
    print(f"worst {rank}:", ", ".join(f"{n['name']}({n['tips']} tips/{n['pieces']} pieces)" for n in worst))

### Are the fragmenting splits supported?

Support of the edge above every extra piece. High support means the tree genuinely places those
tips elsewhere (a label or data problem); low support means the placement is uncertain.

In [ ]:
rows = []
for rank in lib.RANKS:
    for name, size, support in lib.extra_piece_supports(pieces[rank]):
        rows.append((rank, name, size, support))
sup = np.array([r[3] for r in rows if r[3] is not None], dtype=float)
if len(sup):
    print(f"{len(sup)} extra pieces with a support value: median {np.median(sup):.2f}; "
          f"< 0.8: {(sup < 0.8).mean():.1%}; < 0.9: {(sup < 0.9).mean():.1%}; >= 0.95: {(sup >= 0.95).mean():.1%}")
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].hist(sup, bins=20); ax[0].set_title("support of extra pieces")
    sizes = np.array([r[2] for r in rows if r[3] is not None])
    ax[1].scatter(sizes, sup, s=6); ax[1].set_xscale("log"); ax[1].set_xlabel("piece size (tips)"); ax[1].set_ylabel("support")
    plt.show()
else:
    print("no support values in the tree")
with open(OUT / "extra_piece_support.tsv", "w") as f:
    f.write("rank\tname\tpiece_tips\tsupport\n")
    for r in rows:
        f.write("\t".join("" if v is None else str(v) for v in r) + "\n")

## 2. Tips that look misplaced

Tips whose nearest labelled neighbours (>= 8) agree at >= 90% on another name that is **not** a sister
(different parent clade). Compared with how complete / duplicated / well-annotated those genomes are.

In [ ]:
import csv
mapping = {}
with open(P["taxonomy_mapping"]) as f:
    for row in csv.DictReader(f, delimiter="\t"):
        mapping[row["acc"]] = row.get("tax_status", "")
tables = {}
for entry in P["busco_manifests"]:
    if not Path(entry["manifest"]).exists():
        continue
    with open(entry["manifest"]) as f:
        for row in csv.DictReader(f, delimiter="\t"):
            path = Path(row["busco_result"])
            tables[row["genome_acc"]] = path if path.is_absolute() else Path(entry["base"]) / path

def genome_stats(acc):
    path = tables.get(acc)
    if path is None or not Path(path).exists():
        return None
    c = lib.busco_status_counts(path)
    total = sum(c.values()) or 1
    return {"complete": c.get("Complete", 0), "duplicated": c.get("Duplicated", 0), "dup_frac": c.get("Duplicated", 0) / total}

flagged = {}
for rank in ("phylum", "class", "order", "family"):
    rows = lib.misplaced_tips(tree, taxonomy, rank)
    flagged[rank] = rows
    ns = [r for r in rows if not r["sister"]]
    print(f"{rank:7} flagged {len(rows):4d}  (sister-group swaps {len(rows) - len(ns)}, non-sister {len(ns)})")
rows = flagged["order"] + flagged["family"]
non_sister = {r["tip"] for r in rows if not r["sister"]}
all_tips = [t.name for t in tree.get_terminals()]
def summarise(accs, label):
    st = [s for s in (genome_stats(a) for a in accs) if s]
    if not st:
        print(label, "no BUSCO tables"); return
    status = [mapping.get(a, "?") for a in accs]
    print(f"{label:22} n={len(accs):5d}  median complete={np.median([s['complete'] for s in st]):.0f}  "
          f"median duplicated={np.median([s['duplicated'] for s in st]):.0f}  "
          f"median dup fraction={np.median([s['dup_frac'] for s in st]):.3f}  "
          f"taxonomy partial={np.mean([x == 'partial' for x in status]):.1%}")
summarise(sorted(non_sister), "non-sister misplaced")
summarise(all_tips, "all tips")
with open(OUT / "misplaced_tips.tsv", "w") as f:
    f.write("rank\ttip\tlabel\tneighbours_say\tagreement\tn_neighbours\tsister\n")
    for rank, rs in flagged.items():
        for r in rs:
            f.write(f"{rank}\t{r['tip']}\t{r['label']}\t{r['neighbours_say']}\t{r['agreement']:.3f}\t{r['n_neighbours']}\t{r['sister']}\n")
for r in [r for r in flagged['order'] if not r['sister']][:15]:
    print(f"  {r['tip']:28} {r['label']:24} -> {r['neighbours_say']} ({r['agreement']:.0%} of {r['n_neighbours']})")

## 3. RED

Spread of PhyloRank's median RED per named taxon at each rank against the cutoff used for
naming, and how non-monotone the PhyloRank tree is. `assign_taxonomy` raises each node's RED to
at least its parent's and ignores PhyloRank's rewritten branch lengths.

In [ ]:
red = lib.red_by_rank(P["red_tsv"], P["red_dict"])
print(f"{'rank':8}{'cutoff':>8}{'taxa':>6}{'q10':>7}{'median':>8}{'q90':>7}")
for rank in ("phylum", "class", "order", "family", "genus", "species"):
    if rank not in red or not red[rank][1]:
        continue
    cutoff, v = red[rank]; v = np.asarray(v)
    print(f"{rank:8}{cutoff:>8.3f}{len(v):>6}{np.quantile(v, .1):>7.3f}{np.median(v):>8.3f}{np.quantile(v, .9):>7.3f}")
over = lib.rank_overlap(red)
print("\nshare of taxa whose RED is on the wrong side of the neighbouring rank's median:")
for rank, (below, above) in over.items():
    print(f"  {rank:8} below parent-rank median {below:.2f}   above child-rank median {above:.2f}")
fig, ax = plt.subplots(figsize=(8, 3.5))
ranks = [r for r in ("phylum", "class", "order", "family", "genus") if red.get(r) and red[r][1]]
ax.boxplot([red[r][1] for r in ranks]); ax.set_xticks(range(1, len(ranks) + 1)); ax.set_xticklabels(ranks)
ax.plot(range(1, len(ranks) + 1), [red[r][0] for r in ranks], "r_", markersize=30, label="cutoff"); ax.legend(); ax.set_ylabel("median RED per taxon")
plt.show()
red_tree = Phylo.read(P["red_tree"], "newick")
n, lower, neg, worst = lib.red_monotonicity(red_tree)
print(f"\nPhyloRank RED tree: {n} internal nodes with RED, {lower} below their parent, {neg} negative branch lengths (worst {worst:.4f})")
with open(OUT / "red_summary.tsv", "w") as f:
    f.write("rank\tcutoff\ttaxa\tmedian\n")
    for rank, (cutoff, v) in red.items():
        f.write(f"{rank}\t{cutoff}\t{len(v)}\t{np.median(v) if v else ''}\n")